# `src_2` Pipeline Walkthrough — Flattened

This notebook **flattens** the `run_completed_cycle` use case (in
[`src_2/application/reporting.py`](src_2/application/reporting.py)) into a single
linear, runnable story. Each deterministic stage is executed in order, its output
inspected, and the key **Streamlit visualizations** (from
[`src_2/presentation/streamlit_app.py`](src_2/presentation/streamlit_app.py)) are
reproduced inline.

Callout boxes marked **LLM SWAP POINT** show exactly where the recommended
LLM-driven changes plug in (see [`src_2/docs/llm_migration.md`](src_2/docs/llm_migration.md)).

## The flow

```
data/input/sampe_2/*.json
    │  1. ingestion.load_sample2        -> RawCyclePayload
    ▼
    │  2. ingestion.normalize_cycle     -> CanonicalCycleData  (privacy-safe facts)
    │  3. ingestion.build_data_quality_report -> DataQualityReport
    ▼
    │  4. config: registry + budget policy (YAML)
    │  5. analytics.build_scorecards    -> CycleScorecards (5 levels)   [KEEP DETERMINISTIC — arithmetic]
    ▼
    │  6. analytics.build_assessment_bundle -> assessments + evidence   ★ LLM SWAP POINT (CampaignAssessor)
    │  7. analytics.build_budget_scenario   -> BudgetScenario           ★ LLM SWAP POINT (BudgetAllocator)
    ▼
    │  8. intelligence: analyze / synthesize / narrate                  ★ LLM SWAP POINT (already a port)
    ▼
    CompletedCycleReport  ->  Streamlit tabs / JSON export
```

**Principle carried through:** deterministic code *measures* (stages 1–5); the LLM
*judges* (stages 6–8). The arithmetic is never handed to the model.


In [1]:
# --- Setup -------------------------------------------------------------------
import sys, os, json
from pathlib import Path

# Make the repo root (the folder containing `src_2`) importable, regardless of
# where the notebook is launched from.
here = Path.cwd()
for _p in [here, *here.parents]:
    if (_p / "src_2" / "__init__.py").exists():
        ROOT = _p
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

import pandas as pd
import altair as alt
# Emit the native Vega-Lite mimebundle so charts render inline in VS Code / Jupyter
# without a CDN round-trip.
alt.renderers.enable("mimetype")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("repo root:", ROOT)


repo root: /Users/abdelmoo/Desktop/CAPI Analysis/Team2-MarketingExpert


## Chart helpers (lifted from the Streamlit app)

These are the same Altair specs used in `streamlit_app.py`, with the `st.*`
wrappers removed so they render directly in the notebook. The label/colour maps
are copied verbatim from the app.

In [2]:
ACTION_LABELS = {
    "scale": "Scale", "keep_as_test": "Keep as test", "do_not_fund": "Do not fund",
    "insufficient_evidence": "Insufficient evidence", "data_not_ready": "Data not ready",
}
TARGET_LABELS = {
    "achieved": "Achieved", "achieved_with_concerns": "Achieved with concerns",
    "not_achieved": "Not achieved", "insufficient_evidence": "Insufficient evidence",
    "data_not_ready": "Data not ready",
}
ACTION_COLORS = {
    "keep_as_test": "#3D8DFF", "scale": "#2E8B57", "do_not_fund": "#C94C4C",
    "insufficient_evidence": "#D89A2B", "data_not_ready": "#6B7280",
}

def portfolio_chart(campaigns: pd.DataFrame) -> alt.Chart:
    d = campaigns.copy()
    d["campaign_type_label"] = d["campaign_type"].str.replace("_", " ", regex=False).str.title()
    return (alt.Chart(d).mark_circle(opacity=0.82, stroke="white", strokeWidth=1.5).encode(
        x=alt.X("spend:Q", title="Spend (EGP)", scale=alt.Scale(zero=True)),
        y=alt.Y("net_revenue:Q", title="Observed net revenue (EGP)"),
        size=alt.Size("delivered_orders:Q", title="Delivered orders", scale=alt.Scale(range=[100, 1100])),
        color=alt.Color("campaign_type_label:N", title="Campaign type", scale=alt.Scale(scheme="tableau10")),
        tooltip=[alt.Tooltip("campaign_name:N", title="Campaign"),
                 alt.Tooltip("net_roas:Q", title="Net ROAS", format=".2f"),
                 alt.Tooltip("delivered_orders:Q", title="Delivered orders", format=",.0f")],
    ).properties(height=380, title="Portfolio economics — spend vs. observed net revenue"))

def action_count_chart(campaigns: pd.DataFrame) -> alt.Chart:
    counts = (campaigns["next_cycle_action"].map(ACTION_LABELS)
              .value_counts().rename_axis("Action").reset_index(name="Campaigns"))
    return (alt.Chart(counts).mark_bar(cornerRadiusEnd=3).encode(
        y=alt.Y("Action:N", sort="-x", title=None),
        x=alt.X("Campaigns:Q", title="Campaign count", axis=alt.Axis(tickMinStep=1)),
        color=alt.value("#3D8DFF"), tooltip=["Action:N", "Campaigns:Q"],
    ).properties(height=190, title="Next-cycle decisions"))

def benchmark_chart(pack) -> alt.Chart:
    rows = []
    for m in pack.primary_kpis:
        rows += [{"Metric": m.label, "Value type": "Actual", "Value": m.actual},
                 {"Metric": m.label, "Value type": "Benchmark", "Value": m.benchmark}]
    return (alt.Chart(pd.DataFrame(rows)).mark_bar().encode(
        x=alt.X("Value type:N", title=None), y=alt.Y("Value:Q", title=None),
        color=alt.Color("Value type:N", scale=alt.Scale(domain=["Actual", "Benchmark"], range=["#3D8DFF", "#B8BCC4"]), legend=None),
        column=alt.Column("Metric:N", title=None),
    ).properties(height=200, title="Primary KPI vs. benchmark"))

def entity_chart(frame: pd.DataFrame) -> alt.Chart:
    d = frame.copy()
    d["Decision"] = d["next_cycle_action"].map(ACTION_LABELS)
    domain = [ACTION_LABELS[k] for k in ACTION_COLORS]
    colors = [ACTION_COLORS[k] for k in ACTION_COLORS]
    return (alt.Chart(d).mark_circle(opacity=0.82, stroke="white", strokeWidth=1.4).encode(
        x=alt.X("spend:Q", title="Spend (EGP)"), y=alt.Y("net_roas:Q", title="Observed net ROAS"),
        size=alt.Size("observed_conversations:Q", title="Observed conversations", scale=alt.Scale(range=[90, 1000])),
        color=alt.Color("Decision:N", scale=alt.Scale(domain=domain, range=colors)),
        tooltip=[alt.Tooltip("entity_name:N", title="Entity"),
                 alt.Tooltip("net_roas:Q", title="Net ROAS", format=".2f"), "Decision:N"],
    ).properties(height=320, title="Execution-level entities by decision"))

def budget_chart(scenario) -> alt.Chart:
    d = pd.DataFrame([{ "Campaign": a.entity_name, "Budget units": a.budget_units,
                        "Action": ACTION_LABELS[a.action.value]} for a in scenario.allocations])
    d = d[d["Budget units"] > 0].sort_values("Budget units", ascending=False)
    return (alt.Chart(d).mark_bar(cornerRadiusEnd=3, color="#3D8DFF").encode(
        y=alt.Y("Campaign:N", sort="-x", title=None, axis=alt.Axis(labelLimit=230)),
        x=alt.X("Budget units:Q", title="Normalized budget units"),
        tooltip=["Campaign:N", "Action:N", alt.Tooltip("Budget units:Q", format=".2f")],
    ).properties(height=340, title="Illustrative next-cycle budget allocation"))

print("chart helpers ready")


chart helpers ready


---
## Stage 1 — Ingestion: load the raw cycle

[`ingestion.load_sample2`](src_2/ingestion/loaders.py) reads the three raw JSON
files into a frozen `RawCyclePayload`. It validates only the top-level JSON types;
no business logic yet.

In [3]:
from src_2.ingestion import load_sample2
from src_2.paths import INPUT_DIR

raw = load_sample2(INPUT_DIR)
print("input dir:", INPUT_DIR)
print("raw payload type:", type(raw).__name__)
print("raw payload fields:", [f for f in vars(raw)])
_meta = raw.meta
print("meta type:", type(_meta).__name__,
      "| keys:", list(_meta.keys())[:8] if isinstance(_meta, dict) else f"{len(_meta)} records")
print("conversations records:", len(raw.conversations))
print("products records     :", len(raw.products))


input dir: /Users/abdelmoo/Desktop/CAPI Analysis/Team2-MarketingExpert/src_2/data/input/sampe_2
raw payload type: RawCyclePayload
raw payload fields: ['meta', 'conversations', 'products', 'source_directory']
meta type: dict | keys: ['campaigns', 'adsets', 'creatives', 'ads', 'insights']
conversations records: 788
products records     : 105


## Stage 2 — Normalize into privacy-safe canonical facts

[`ingestion.normalize_cycle`](src_2/ingestion/normalizer.py) turns messy raw JSON
into clean fact tables (`campaigns`, `adsets`, `ads`, `creatives`, `media_daily`,
`conversations`, `order_lines`, `products`). **Raw message text, phone numbers, and
customer names are deliberately dropped here** — they never travel downstream.

In [4]:
from src_2.ingestion import normalize_cycle

canonical = normalize_cycle(raw)
print("cycle window:", canonical.cycle_start, "->", canonical.cycle_end)
for name in ["campaigns", "adsets", "ads", "creatives", "media_daily", "conversations", "order_lines", "products"]:
    frame = getattr(canonical, name, None)
    if frame is not None:
        print(f"{name:14s} rows={len(frame):5d}  cols={list(frame.columns)[:6]}")


cycle window: 2025-12-30 00:00:00 -> 2026-06-27 00:00:00
campaigns      rows=   12  cols=['campaign_id', 'campaign_name', 'objective', 'campaign_type', 'start_date', 'end_date']
adsets         rows=   26  cols=['adset_id', 'adset_name', 'campaign_id', 'audience_type', 'optimization_goal', 'billing_event']
ads            rows=   40  cols=['ad_id', 'ad_name', 'adset_id', 'campaign_id', 'start_date', 'end_date']
creatives      rows=   30  cols=['creative_id', 'creative_name', 'theme', 'angle', 'status', 'object_story_spec.page_id']
media_daily    rows= 1243  cols=['ad_id', 'adset_id', 'campaign_id', 'date_start', 'date_stop', 'impressions']
conversations  rows=  788  cols=['conversation_id', 'campaign_id', 'adset_id', 'ad_id', 'creative_id', 'audience_type']
order_lines    rows= 1081  cols=['conversation_id', 'campaign_id', 'adset_id', 'ad_id', 'creative_id', 'audience_type']
products       rows=  105  cols=['product_id', 'name', 'category', 'price', 'tags', 'contents']


In [5]:
# A peek at the canonical conversations table (privacy-safe: outcomes only, no message text)
canonical.conversations.head(5)


,conversation_id,campaign_id,adset_id,ad_id,creative_id,audience_type,source_platform,customer_id,language,cycle_number,started_at,last_message_at,conversation_minutes,message_count,inbound_messages,outbound_messages,outcome_type,order_id,has_order,is_delivered,is_refunded,is_cancelled,is_ghosted,is_negative,is_open_or_pending,is_repeat_cycle,is_repeat_delivered,gross_order_value,delivered_revenue,net_revenue,refunded_amount,cancelled_value,pending_value
0,conv_001,120209876543220006,120209876543240011,120209876543210017,120209876543230007,lookalike,meta_ctwa,cust_001,ar,2,2026-03-06 20:47:04+00:00,2026-03-06 21:00:04+00:00,13.0,6,4,2,stuck_pending,ord_001,True,False,False,False,False,False,True,True,False,947.0,0.0,0.0,0.0,0.0,947.0
1,conv_002,None,None,None,None,None,organic,cust_476,ar,3,2026-06-21 10:18:24+00:00,2026-06-21 14:21:24+00:00,243.0,8,4,4,cancelled,ord_002,True,False,False,True,False,True,False,True,False,297.0,0.0,0.0,0.0,297.0,0.0
2,conv_003,120209876543220012,120209876543240025,120209876543210039,120209876543230020,custom,meta_ctwa,cust_003,en,3,2026-06-25 11:38:39+00:00,2026-06-25 11:46:39+00:00,8.0,5,3,2,delivered,ord_003,True,True,False,False,False,False,False,True,True,616.0,616.0,616.0,0.0,0.0,0.0
3,conv_004,None,None,None,None,None,direct,cust_029,en,3,2026-06-10 14:03:53+00:00,2026-06-10 14:33:53+00:00,30.0,8,4,4,delivered,ord_004,True,True,False,False,False,False,False,True,True,2724.0,2724.0,2724.0,0.0,0.0,0.0
4,conv_005,None,None,None,None,None,organic,cust_005,ar,2,2026-04-18 15:28:39+00:00,2026-04-18 15:31:39+00:00,3.0,4,2,2,ghosted,None,False,False,False,False,True,True,False,True,False,0.0,0.0,0.0,0.0,0.0,0.0


## Stage 3 — Data-quality report

[`ingestion.build_data_quality_report`](src_2/ingestion/data_quality.py)
reconciles Meta-attributed conversation *starts* against the supplied WhatsApp
conversation rows and sets an `EvidenceStatus`. This is the "how much can we trust
this?" layer that gates every decision downstream.

In [6]:
from src_2.ingestion import build_data_quality_report

quality = build_data_quality_report(canonical)
print("evidence status         :", getattr(quality.status, "value", quality.status))
print("meta conversation starts:", f"{quality.meta_conversation_starts:,}")
print("observed WhatsApp rows  :", f"{quality.observed_meta_whatsapp_conversations:,}")
print("reconciliation ratio    :", f"{quality.reconciliation_ratio:.2%}")
print("\nwarnings:")
for w in quality.warnings:
    print(" -", w)


evidence status         : limited_evidence
meta conversation starts: 116,098
observed WhatsApp rows  : 617
reconciliation ratio    : 0.53%

warnings:
 - Meta-attributed conversation starts and supplied WhatsApp records are not reconciled populations; outcome rankings describe the observed sample.


## Stage 4 — Load business policy (YAML config)

Policy lives in human-editable YAML ([`config/`](src_2/config/)) validated by
Pydantic. `campaign_types.yaml` defines each type's business job, primary KPI,
guardrails and allocation metric; `budget_policy.poc.yaml` defines the 100-unit
envelope rules.

In [7]:
from src_2.infrastructure.configuration import load_campaign_type_registry, load_budget_policy

registry = load_campaign_type_registry()
policy = load_budget_policy()
_types = registry.campaign_types
_keys = list(_types.keys()) if isinstance(_types, dict) else _types
print("campaign types configured:", [getattr(k, "value", k) for k in _keys])
print("budget units:", policy.budget_units)


campaign types configured: ['awareness', 'always_on', 'promotional', 'seasonal', 'experimental', 'launch', 'scale', 'retention']
budget units: 100.0


## Stage 5 — Scorecards (deterministic KPI math) &nbsp; 🔒 KEEP DETERMINISTIC

[`analytics.build_scorecards`](src_2/analytics/aggregations.py) aggregates media,
outcomes and products into five entity levels and computes ~25 derived KPIs.

> **🔒 Why this stays deterministic:** this is where hundreds of fact rows are
> summed and divided. Handing this arithmetic to an LLM produces non-reproducible,
> unauditable numbers. The migration plan keeps this stage as-is and feeds the
> *computed* results to the model as evidence.

In [8]:
from src_2.analytics import build_scorecards

raw_scorecards = build_scorecards(canonical)
campaign_kpis = raw_scorecards.campaign[
    ["campaign_name", "campaign_type", "spend", "net_revenue", "net_roas",
     "observed_conversations", "delivered_orders"]
].sort_values("spend", ascending=False)
campaign_kpis


,campaign_name,campaign_type,spend,net_revenue,net_roas,observed_conversations,delivered_orders
0,Always-On Premium Acquisition,always_on,140783.91,65824.0,0.467553,111,52
8,Summer Premium Launch,launch,68595.84,70521.0,1.028065,97,64
5,Ramadan Iftar Premium Bundles,seasonal,57095.26,76998.0,1.348588,84,42
4,Ramadan Suhoor Specials,seasonal,32293.38,38002.0,1.176774,64,41
9,Lookalike Scale Cycle 3,scale,31465.87,37936.0,1.205624,36,23
10,Mid-Year Sale,promotional,17552.49,23431.0,1.334910,34,20
6,Eid Gifting Premium,seasonal,14976.24,64990.0,4.339540,47,28
2,Pre-Ramadan Bundle Promo,promotional,13958.72,28905.0,2.070749,46,21
11,Summer Retention Push,retention,9006.65,4722.0,0.524279,16,7
7,Post-Eid Lookalike Test,experimental,7762.12,42145.0,5.429573,63,27


## Stage 6 — Assessment: target status + funding decision &nbsp; ★ LLM SWAP POINT

[`analytics.build_assessment_bundle`](src_2/analytics/assessment_engine.py) resolves
benchmarks and assigns each campaign a `target_status` and a `next_cycle_action`
(scale / keep-as-test / do-not-fund / insufficient-evidence). It also enriches the
scorecards with those decision columns.

> **★ LLM SWAP POINT — `CampaignAssessor` port.** In the recommended design this
> whole stage sits behind a new `CampaignAssessor` Protocol. The deterministic rules
> here become the default + fallback; an `OpenAICampaignAssessor` would receive the
> evidence pack + the campaign-type config and emit the **same `CampaignAssessment`
> contract** — so nothing downstream changes. See `docs/llm_migration.md` §Step 1.

In [9]:
from src_2.analytics import build_assessment_bundle

cycle_id = f"cycle_{canonical.cycle_start.date()}_{canonical.cycle_end.date()}"
bundle = build_assessment_bundle(cycle_id, raw_scorecards, registry, quality)

# `bundle.scorecards` are the ENRICHED scorecards (now carry target_status / next_cycle_action)
assessed = bundle.scorecards.campaign
assessed[["campaign_name", "campaign_type", "target_status", "next_cycle_action"]]


,campaign_name,campaign_type,target_status,next_cycle_action
0,Always-On Premium Acquisition,always_on,not_achieved,do_not_fund
1,Awareness Boost January,awareness,not_achieved,keep_as_test
2,Pre-Ramadan Bundle Promo,promotional,achieved_with_concerns,keep_as_test
3,January Trial Bundle Test,experimental,not_achieved,keep_as_test
4,Ramadan Suhoor Specials,seasonal,not_achieved,keep_as_test
5,Ramadan Iftar Premium Bundles,seasonal,not_achieved,do_not_fund
6,Eid Gifting Premium,seasonal,achieved_with_concerns,keep_as_test
7,Post-Eid Lookalike Test,experimental,achieved_with_concerns,keep_as_test
8,Summer Premium Launch,launch,achieved_with_concerns,keep_as_test
9,Lookalike Scale Cycle 3,scale,not_achieved,keep_as_test


In [10]:
# Reproduce the "Next-cycle decisions" bar from the Executive tab
action_count_chart(assessed)


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [11]:
# Reproduce the Executive "Portfolio economics" scatter
portfolio_chart(assessed)


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


### One campaign's evidence pack + benchmark chart

Each `CampaignEvidencePack` is the exact payload the LLM would receive at the swap
point — computed KPIs and benchmarks only, no raw customer data.

In [12]:
pack = bundle.evidence_packs[0]
print("campaign   :", pack.campaign_name, f"({pack.campaign_type.value})")
print("business job:", pack.business_job)
print("primary KPIs:")
for m in pack.primary_kpis:
    print(f"   {m.label:32s} actual={m.actual}  benchmark={m.benchmark}  passed={m.passed}")
benchmark_chart(pack)


campaign   : Always-On Premium Acquisition (always_on)
business job: Acquire steady sales with stable economics.
primary KPIs:
   Net return on ad spend           actual=0.46755342993386106  benchmark=1.205623744075724  passed=False


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


### Execution-level drill-down (adset / audience / ad / creative)

The deep-dive tab plots child entities coloured by decision. Same `entity_chart`
spec here, for the first campaign's adsets.

In [13]:
first_id = str(assessed.iloc[0]["campaign_id"])
adsets = bundle.scorecards.by_level("adset")
adsets = adsets[adsets["campaign_id"].eq(first_id)]
entity_chart(adsets) if len(adsets) else print("no adsets for this campaign")


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## Stage 7 — Budget scenario &nbsp; ★ LLM SWAP POINT

[`analytics.build_budget_scenario`](src_2/analytics/allocation.py) spreads 100
normalized units across campaign-type envelopes and, within each type, by a single
allocation KPI. Blocked campaigns leave their budget unallocated.

> **★ LLM SWAP POINT — `BudgetAllocator` port.** Recommended: let the LLM decide
> *strategy and rationale*, but emit weights that a thin deterministic step
> normalizes to exactly `budget_units` (so the numbers always sum correctly). Output
> stays the `BudgetScenario` contract. See `docs/llm_migration.md` §Allocation Caveat.

In [14]:
from src_2.analytics import build_budget_scenario

scenario = build_budget_scenario(
    cycle_id, bundle.scorecards.campaign, bundle.assessments, registry, policy, quality
)
print("total units:", scenario.total_budget_units, " | unallocated:", round(scenario.unallocated_units, 2))
budget_chart(scenario)


total units: 100.0  | unallocated: 37.24


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## Stage 8 — Intelligence: analyze / synthesize / narrate &nbsp; ★ LLM SWAP POINT (already a port)

The narrative layer is **already** behind ports
([`application/ports.py`](src_2/application/ports.py)). Below we run the
**deterministic** adapters. To go LLM here you swap in
`OpenAICampaignAnalyst / OpenAIPortfolioSynthesizer / OpenAIReportNarrator` — no
new code, just injection (this is Rollout Step 1 in the migration doc).

In [15]:
from src_2.intelligence import (
    DeterministicCampaignAnalyst,
    DeterministicPortfolioSynthesizer,
    DeterministicReportNarrator,
)

analyst = DeterministicCampaignAnalyst()
insights = [analyst.analyze(p) for p in bundle.evidence_packs]
portfolio = DeterministicPortfolioSynthesizer().synthesize(bundle.assessments, insights)
report_text = DeterministicReportNarrator().narrate(portfolio, scenario)

print("=== Executive summary ===\n", report_text.executive_summary, "\n")
print("=== First campaign insight ===")
print("assessment:", insights[0].target_assessment)
print("drivers   :", insights[0].performance_drivers)
print("next test :", insights[0].next_controlled_test)


=== Executive summary ===
 This completed-cycle report assigns 62.8 of 100 illustrative units. The leading candidates are Eid Gifting Premium (20.1 units), Summer Premium Launch (17.1 units), Lookalike Scale Cycle 3 (7.8 units). 

=== First campaign insight ===
assessment: The always on campaign did not meet its primary Net return on ad spend reference: 0.47 versus 1.21.
drivers   : ['Delivered-order rate: 0.47', 'Cost per delivered order: 2,707.38', 'Observed WhatsApp conversations: 111.00']
next test : Test Broad and Always-on premium acquisition default in matched cells with the same budget, timing, offer, and optimization settings.


In [16]:
# The swap is literally this (commented out — needs OPENAI_API_KEY in .env):
#
# from src_2.intelligence import (
#     OpenAICampaignAnalyst, OpenAIPortfolioSynthesizer, OpenAIReportNarrator,
# )
# insights   = [OpenAICampaignAnalyst().analyze(p) for p in bundle.evidence_packs]
# portfolio  = OpenAIPortfolioSynthesizer().synthesize(bundle.assessments, insights)
# report_text = OpenAIReportNarrator().narrate(portfolio, scenario)
print("LLM swap for the narrative layer is a drop-in — same contracts, no downstream changes.")


LLM swap for the narrative layer is a drop-in — same contracts, no downstream changes.


## The whole thing in one call

Everything above is exactly what `run_completed_cycle` does internally. In practice
you call it once; the flattening above is just to see each stage.

In [17]:
from src_2.application import run_completed_cycle

report = run_completed_cycle()          # deterministic default; add ports to go LLM
print("cycle:", report.manifest.cycle_id)
print("campaigns:", len(report.scorecards.campaign),
      "| assessments:", len(report.assessments),
      "| insights:", len(report.insights),
      "| allocations:", len(report.budget_scenario.allocations))

# to inject the LLM end-to-end (needs OPENAI_API_KEY):
# from src_2.intelligence import OpenAICampaignAnalyst, OpenAIPortfolioSynthesizer, OpenAIReportNarrator
# report = run_completed_cycle(
#     campaign_analyst=OpenAICampaignAnalyst(),
#     portfolio_synthesizer=OpenAIPortfolioSynthesizer(),
#     report_narrator=OpenAIReportNarrator(),
# )


cycle: cycle_2025-12-30_2026-06-27
campaigns: 12 | assessments: 12 | insights: 12 | allocations: 12


---
## Where the LLM plugs in — summary

| Stage | Function | Status today | Recommended change |
|---|---|---|---|
| 1–2 Ingestion | `load_sample2`, `normalize_cycle` | deterministic | 🔒 keep — parsing/PII |
| 3 Data quality | `build_data_quality_report` | deterministic | optional LLM *interpretation* only |
| 5 Scorecards | `build_scorecards` | deterministic | 🔒 keep — arithmetic |
| 6 Assessment | `build_assessment_bundle` | hardcoded call | ★ new `CampaignAssessor` port |
| 7 Budget | `build_budget_scenario` | hardcoded call | ★ new `BudgetAllocator` port (normalize units deterministically) |
| 8 Narrative | `intelligence.*` | **already a port** | ★ inject `OpenAI*` adapters (ships today) |

**Guardrails required once the LLM makes decisions:** semantic post-validation +
deterministic fallback, `temperature=0` + caching, and an eval harness that diffs
LLM decisions against the deterministic baseline. Full detail in
[`src_2/docs/llm_migration.md`](src_2/docs/llm_migration.md).
